# Hands-on 24: Klasifikasi Status Perangkat (normal, warning, fault)

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 8.

Dataset multi-kelas dari `room_classification_dataset.csv` (split time-aware 70/15/15),
lalu dua model: Logistic Regression (dengan scaler) dan Random Forest Classifier.

## Tujuan

Pada latihan ini, pembaca akan membangun model klasifikasi untuk memprediksi status perangkat:

- normal
- warning
- fault

Setelah selesai, pembaca diharapkan mampu:

- membaca dataset klasifikasi multi-kelas,
- melatih Logistic Regression,
- melatih Random Forest Classifier,
- mengukur accuracy, precision, recall, F1, dan confusion matrix,
- dan menafsirkan kesalahan klasifikasi.

Ini sangat relevan untuk IoT karena banyak keputusan operasional memang berbentuk status diskret seperti ini. Referensi juga menunjukkan bahwa Random Forest Classifier dan Logistic Regression banyak dipakai pada sistem IoT dan evaluasinya lazim memakai confusion matrix dan akurasi.

## Contoh implementasi nyata

Smart room monitoring

Status: ● normal ● warning bila suhu/CO₂ mulai mendekati ambang ● fault bila kondisi buruk atau sistem gagal ventilasi

Predictive maintenance

Status: ● normal ● warning bila getaran mulai naik ● fault bila pola mesin jelas abnormal

Smart energy

Status: ● normal ● warning saat beban tinggi ● fault saat lonjakan ekstrem atau kondisi tidak wajar

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /downloads/iot-ml-book


## 1. Siapkan Dataset Multi-Kelas

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/prepare_multiclass_status_dataset.py"], capture_output=True, text=True)
print(r.stdout)

## 2. Logistic Regression

In [3]:
r = subprocess.run([sys.executable, "scripts/classification_logreg_status.py"], capture_output=True, text=True)
print(r.stdout)

=== LOGISTIC REGRESSION - VALIDATION ===
Accuracy: 0.9938271604938271

Confusion Matrix:
[[307   0   0]
 [  2   9   0]
 [  0   0   6]]

Classification Report:
              precision    recall  f1-score   support

       fault       1.00      1.00      1.00         6
      normal       0.99      1.00      1.00       307
     warning       1.00      0.82      0.90        11

    accuracy                           0.99       324
   macro avg       1.00      0.94      0.97       324
weighted avg       0.99      0.99      0.99       324


=== LOGISTIC REGRESSION - TEST ===
Accuracy: 0.9907407407407407

Confusion Matrix:
[[305   0   0]
 [  3  11   0]
 [  0   0   5]]

Classification Report:
              precision    recall  f1-score   support

       fault       1.00      1.00      1.00         5
      normal       0.99      1.00      1.00       305
     warning       1.00      0.79      0.88        14

    accuracy                           0.99       324
   macro avg       1.00      0.93 

## 3. Random Forest Classifier

In [4]:
r = subprocess.run([sys.executable, "scripts/classification_rf_status.py"], capture_output=True, text=True)
print(r.stdout)

=== RANDOM FOREST CLASSIFIER - VALIDATION ===
Accuracy: 0.9907407407407407

Confusion Matrix:
[[306   1   0]
 [  2   9   0]
 [  0   0   6]]

Classification Report:
              precision    recall  f1-score   support

       fault       1.00      1.00      1.00         6
      normal       0.99      1.00      1.00       307
     warning       0.90      0.82      0.86        11

    accuracy                           0.99       324
   macro avg       0.96      0.94      0.95       324
weighted avg       0.99      0.99      0.99       324


=== RANDOM FOREST CLASSIFIER - TEST ===
Accuracy: 0.9845679012345679

Confusion Matrix:
[[304   1   0]
 [  4  10   0]
 [  0   0   5]]

Classification Report:
              precision    recall  f1-score   support

       fault       1.00      1.00      1.00         5
      normal       0.99      1.00      0.99       305
     warning       0.91      0.71      0.80        14

    accuracy                           0.98       324
   macro avg       0.97 

## Panduan Lengkap dari Buku

_Bagian ini dikutip dari buku PDF (sumber kebenaran praktikum) dan diformat ulang agar mudah dibaca. Kode asli buku tidak diubah; bagian instalasi/persiapan lingkungan diringkas karena sudah dijelaskan di README._

**3. Dataset yang digunakan**

Gunakan dataset klasifikasi multi-kelas, misalnya:

data/synthetic/room_classification_dataset.csv

dengan kolom:

- timestamp
- temperature_c
- humidity_pct
- co2_ppm
- power_watt
- status_label

Jika sebelumnya Anda baru membuat target biner normal vs abnormal, untuk latihan ini kita gunakan label tiga kelas.

**4. Menyiapkan dataset multi-kelas**

Buat file scripts/prepare_multiclass_status_dataset.py:

```python
   from pathlib import Path
   import pandas as pd
   import numpy as np
   INPUT_PATH = Path("data/synthetic/room_classification_dataset.csv")
   OUTPUT_DIR = Path("data/processed/classification_status")
   OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
   df = pd.read_csv(INPUT_PATH)
   df["timestamp"] = pd.to_datetime(df["timestamp"])
   df = df.sort_values("timestamp").reset_index(drop=True)
   # Pastikan label ada
   if "status_label" not in df.columns:
       def make_label(row):
           if row["temperature_c"] > 33 or row["co2_ppm"] > 1400 or
   row["power_watt"] > 280:
               return "fault"
           elif row["temperature_c"] > 29 or row["co2_ppm"] > 900 or
   row["power_watt"] > 200:
               return "warning"
         return "normal"
     df["status_label"] = df.apply(make_label, axis=1)
# Fitur waktu
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
# Lag dan rolling sederhana
df["temp_lag_1"] = df["temperature_c"].shift(1)
df["co2_lag_1"] = df["co2_ppm"].shift(1)
df["power_lag_1"] = df["power_watt"].shift(1)
df["temp_ma_3"] = df["temperature_c"].rolling(3).mean()
df["co2_ma_3"] = df["co2_ppm"].rolling(3).mean()
df["power_ma_3"] = df["power_watt"].rolling(3).mean()
df = df.dropna().reset_index(drop=True)
feature_cols = [
    "temperature_c", "humidity_pct", "co2_ppm", "power_watt",
    "hour", "day_of_week", "is_weekend",
    "temp_lag_1", "co2_lag_1", "power_lag_1",
    "temp_ma_3", "co2_ma_3", "power_ma_3"
]
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)
train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()
train_df.to_csv(OUTPUT_DIR / "train.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "validation.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "test.csv", index=False)
train_df[feature_cols].to_csv(OUTPUT_DIR / "X_train.csv", index=False)
train_df[["status_label"]].to_csv(OUTPUT_DIR / "y_train.csv",
index=False)
val_df[feature_cols].to_csv(OUTPUT_DIR / "X_val.csv", index=False)
val_df[["status_label"]].to_csv(OUTPUT_DIR / "y_val.csv", index=False)
test_df[feature_cols].to_csv(OUTPUT_DIR / "X_test.csv", index=False)
test_df[["status_label"]].to_csv(OUTPUT_DIR / "y_test.csv",
index=False)
print("Distribusi label:")
print(df["status_label"].value_counts())
print("File disimpan di:", OUTPUT_DIR)
```

Jalankan:

```python
python scripts/prepare_multiclass_status_dataset.py
```

**5. Logistic Regression untuk multi-kelas**

Buat file scripts/classification_logreg_status.py:

```python
from pathlib import Path
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix,
accuracy_score
DATA_DIR = Path("data/processed/classification_status")
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze()
X_val = pd.read_csv(DATA_DIR / "X_val.csv")
y_val = pd.read_csv(DATA_DIR / "y_val.csv").squeeze()
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze()
model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)
# Validasi
y_val_pred = model.predict(X_val)
print("=== LOGISTIC REGRESSION - VALIDATION ===")
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred, labels=["normal", "warning",
"fault"]))
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred, zero_division=0))
# Test
y_test_pred = model.predict(X_test)
print("\n=== LOGISTIC REGRESSION - TEST ===")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred, labels=["normal",
"warning", "fault"]))
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, zero_division=0))
```

**6. Random Forest Classifier untuk multi-kelas**

Buat file scripts/classification_rf_status.py:

```python
from pathlib import Path
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix,
accuracy_score
DATA_DIR = Path("data/processed/classification_status")
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze()
X_val = pd.read_csv(DATA_DIR / "X_val.csv")
y_val = pd.read_csv(DATA_DIR / "y_val.csv").squeeze()
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze()
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
# Validasi
y_val_pred = model.predict(X_val)
print("=== RANDOM FOREST CLASSIFIER - VALIDATION ===")
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred, labels=["normal", "warning",
"fault"]))
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred, zero_division=0))
# Test
y_test_pred = model.predict(X_test)
print("\n=== RANDOM FOREST CLASSIFIER - TEST ===")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred, labels=["normal",
"warning", "fault"]))
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, zero_division=0))
```

Apa yang harus dilihat?

- apakah warning sering salah dikira normal,
- apakah fault tertangkap baik,
- apakah model terlalu percaya diri pada kelas mayoritas.

Confusion matrix sangat penting untuk ini. Referensi juga menekankan bahwa confusion matrix membantu melihat jumlah kesalahan dan false positives pada klasifikasi IoT.

## Penutup

Kedua model mencapai accuracy sekitar 0.98-0.99 pada validasi dan test. Kelas minoritas
(warning/fault) adalah bagian tersulit; confusion matrix menunjukkan kesalahan utama ada
di batas normal-warning. Perbandingan formal dengan kurva ROC/PR ada di Hands-on 25.